## Cell 1 — Import library

In [2]:
import pandas as pd
from pathlib import Path
import pm4py

## Cell 2 — Cek versi PM4Py

In [3]:
print("PM4Py version:", pm4py.__version__)

PM4Py version: 2.7.22.4


## Cell 3 — Tentukan folder project

In [4]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Output figures dir:", OUTPUT_FIGURES_DIR)

Project root: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL
Processed dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed
Output figures dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\figures


## Cell 4 — Baca compact log

In [5]:
COMPACT_LOG_PATH = PROCESSED_DIR / "04_compact_log_process_ready.csv"

compact_log = pd.read_csv(
    COMPACT_LOG_PATH,
    parse_dates=["timestamp"]
)

print("Ukuran compact_log:", compact_log.shape)
display(compact_log.head())

print("\nJumlah mahasiswa:", compact_log["case_id"].nunique())
print("Jumlah activity_mapped unik:", compact_log["activity_mapped"].nunique())
print("Timestamp awal:", compact_log["timestamp"].min())
print("Timestamp akhir:", compact_log["timestamp"].max())

Ukuran compact_log: (116611, 18)


,case_id,activity,timestamp,kelas,Component,Event context,Description,Origin,IP address,nim,nama,nilai_total,nilai_indeks,label_performance,activity_mapped,activity_category,event_order,compact_event_order
0,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 13:30:09,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.204,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Course Viewed,course,1,1
1,ADITYA PUTRA PERMANA,Course module viewed,2025-09-18 14:24:42,IF-49-02,File,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Material Viewed,material,4,2
2,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 15:41:24,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Course Viewed,course,5,3
3,ADITYA PUTRA PERMANA,Course module viewed,2025-09-18 15:41:57,IF-49-02,File,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Material Viewed,material,6,4
4,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 16:40:55,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,36.72.198.4,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Course Viewed,course,7,5



Jumlah mahasiswa: 89
Jumlah activity_mapped unik: 25
Timestamp awal: 2025-09-16 09:49:33
Timestamp akhir: 2026-01-16 22:18:34


## Cell 5 — Pilih kolom inti process mining

In [6]:
pm_df = compact_log[[
    "case_id",
    "activity_mapped",
    "timestamp"
]].copy()

pm_df = pm_df.rename(columns={
    "case_id": "case:concept:name",
    "activity_mapped": "concept:name",
    "timestamp": "time:timestamp"
})

pm_df = pm_df.sort_values(
    by=["case:concept:name", "time:timestamp"]
).reset_index(drop=True)

display(pm_df.head(20))

print("Ukuran pm_df:", pm_df.shape)
print("Jumlah case:", pm_df["case:concept:name"].nunique())
print("Jumlah activity:", pm_df["concept:name"].nunique())

,case:concept:name,concept:name,time:timestamp
0,ADITYA PUTRA PERMANA,Course Viewed,2025-09-18 13:30:09
1,ADITYA PUTRA PERMANA,Material Viewed,2025-09-18 14:24:42
2,ADITYA PUTRA PERMANA,Course Viewed,2025-09-18 15:41:24
3,ADITYA PUTRA PERMANA,Material Viewed,2025-09-18 15:41:57
4,ADITYA PUTRA PERMANA,Course Viewed,2025-09-18 16:40:55
5,ADITYA PUTRA PERMANA,Quiz Module Viewed,2025-09-18 16:50:56
6,ADITYA PUTRA PERMANA,Course Viewed,2025-09-18 16:51:17
7,ADITYA PUTRA PERMANA,Material Viewed,2025-09-18 17:36:04
8,ADITYA PUTRA PERMANA,Course Viewed,2025-09-18 20:36:54
9,ADITYA PUTRA PERMANA,Material Viewed,2025-09-18 20:37:38


Ukuran pm_df: (116611, 3)
Jumlah case: 89
Jumlah activity: 25


## Cell 6 — Format dataframe untuk PM4Py

In [7]:
pm_df = pm4py.format_dataframe(
    pm_df,
    case_id="case:concept:name",
    activity_key="concept:name",
    timestamp_key="time:timestamp"
)

print("Dataframe berhasil diformat untuk PM4Py")
print(pm_df.dtypes)
display(pm_df.head())

Dataframe berhasil diformat untuk PM4Py
case:concept:name                 string
concept:name                      string
time:timestamp       datetime64[us, UTC]
@@index                            int64
@@case_index                       int64
dtype: object


,case:concept:name,concept:name,time:timestamp,@@index,@@case_index
0,ADITYA PUTRA PERMANA,Course Viewed,2025-09-18 13:30:09+00:00,0,0
1,ADITYA PUTRA PERMANA,Material Viewed,2025-09-18 14:24:42+00:00,1,0
2,ADITYA PUTRA PERMANA,Course Viewed,2025-09-18 15:41:24+00:00,2,0
3,ADITYA PUTRA PERMANA,Material Viewed,2025-09-18 15:41:57+00:00,3,0
4,ADITYA PUTRA PERMANA,Course Viewed,2025-09-18 16:40:55+00:00,4,0


## Cell 7 — Convert dataframe ke event log PM4Py

In [8]:
event_log_pm4py = pm4py.convert_to_event_log(pm_df)

print("Event log PM4Py berhasil dibuat")
print("Jumlah trace/case:", len(event_log_pm4py))

Event log PM4Py berhasil dibuat
Jumlah trace/case: 89


## Cell 8 — Cek contoh trace pertama

In [9]:
first_trace = event_log_pm4py[0]

print("Jumlah event pada trace pertama:", len(first_trace))
print("Case ID trace pertama:", first_trace.attributes.get("concept:name"))

print("\n10 aktivitas pertama pada trace pertama:")
for event in first_trace[:10]:
    print("-", event["concept:name"], "|", event["time:timestamp"])

Jumlah event pada trace pertama: 1979
Case ID trace pertama: ADITYA PUTRA PERMANA

10 aktivitas pertama pada trace pertama:
- Course Viewed | 2025-09-18 13:30:09+00:00
- Material Viewed | 2025-09-18 14:24:42+00:00
- Course Viewed | 2025-09-18 15:41:24+00:00
- Material Viewed | 2025-09-18 15:41:57+00:00
- Course Viewed | 2025-09-18 16:40:55+00:00
- Quiz Module Viewed | 2025-09-18 16:50:56+00:00
- Course Viewed | 2025-09-18 16:51:17+00:00
- Material Viewed | 2025-09-18 17:36:04+00:00
- Course Viewed | 2025-09-18 20:36:54+00:00
- Material Viewed | 2025-09-18 20:37:38+00:00


## Cell 9 — Discovery Process Tree dengan Inductive Miner

In [10]:
process_tree = pm4py.discover_process_tree_inductive(event_log_pm4py)

print("Process tree berhasil ditemukan.")
print(process_tree)

Process tree berhasil ditemukan.
*( +( X( tau, *( 'Assignment Viewed', tau ) ), X( tau, ->( 'Assignment Remove Confirmation Viewed', 'Assignment Status Updated', 'Assignment Submission Removed' ), +( X( tau, *( 'Material Viewed', tau ) ), X( tau, ->( +( X( tau, *( 'Forum Viewed', tau ) ), X( tau, ->( *( +( X( tau, 'Assignment File Uploaded' ), X( 'Assignment Submission Created', +( X( tau, 'Assignment Form Viewed' ), X( tau, ->( X( tau, 'Assignment Submitted' ), X( tau, 'Assignment Submission Updated' ) ), ->( X( 'Comment Created', *( ->( X( tau, 'Quiz Started' ), X( tau, +( X( tau, *( 'Quiz Viewed', tau ) ), X( tau, ->( X( tau, +( X( tau, 'Quiz Module Viewed' ), X( tau, +( X( tau, *( 'Quiz Summary Viewed', tau ) ), X( tau, +( X( tau, 'Course Viewed' ), X( tau, +( X( tau, *( 'Quiz Auto-saved', tau ) ), X( tau, *( 'Quiz Updated', tau ) ) ) ) ) ) ) ) ) ), X( tau, 'Assignment Status Viewed', 'Quiz Submitted', 'Quiz Access Prevented' ) ) ) ) ) ), tau ) ), X( tau, 'Comment Deleted' ) ) ) ) 

## Cell 10 — Simpan visual process tree

In [11]:
PROCESS_TREE_PATH = OUTPUT_FIGURES_DIR / "06_process_tree_inductive_miner.png"

pm4py.save_vis_process_tree(
    process_tree,
    PROCESS_TREE_PATH
)

print("Gambar process tree berhasil disimpan ke:")
print(PROCESS_TREE_PATH)

Gambar process tree berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\figures\06_process_tree_inductive_miner.png


## Cell 11 — Discovery Petri Net dengan Inductive Miner

In [12]:
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(event_log_pm4py)

print("Petri net berhasil ditemukan.")
print("Jumlah places:", len(net.places))
print("Jumlah transitions:", len(net.transitions))
print("Initial marking:", initial_marking)
print("Final marking:", final_marking)

Petri net berhasil ditemukan.
Jumlah places: 70
Jumlah transitions: 103
Initial marking: ['source:1']
Final marking: ['sink:1']


## Cell 12 — Simpan visual Petri net

In [13]:
PETRI_NET_PATH = OUTPUT_FIGURES_DIR / "06_petri_net_inductive_miner.png"

pm4py.save_vis_petri_net(
    net,
    initial_marking,
    final_marking,
    PETRI_NET_PATH
)

print("Gambar Petri net berhasil disimpan ke:")
print(PETRI_NET_PATH)

Gambar Petri net berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\figures\06_petri_net_inductive_miner.png


## Cell 13 — Simpan ringkasan model

In [14]:
model_summary = pd.DataFrame({
    "model": ["Process Tree", "Petri Net"],
    "jumlah_case": [
        pm_df["case:concept:name"].nunique(),
        pm_df["case:concept:name"].nunique()
    ],
    "jumlah_event": [
        len(pm_df),
        len(pm_df)
    ],
    "jumlah_activity": [
        pm_df["concept:name"].nunique(),
        pm_df["concept:name"].nunique()
    ],
    "jumlah_places": [
        None,
        len(net.places)
    ],
    "jumlah_transitions": [
        None,
        len(net.transitions)
    ]
})

display(model_summary)

MODEL_SUMMARY_PATH = OUTPUT_TABLES_DIR / "06_process_discovery_model_summary.csv"
model_summary.to_csv(MODEL_SUMMARY_PATH, index=False)

print("Ringkasan model berhasil disimpan ke:")
print(MODEL_SUMMARY_PATH)

,model,jumlah_case,jumlah_event,jumlah_activity,jumlah_places,jumlah_transitions
0,Process Tree,89,116611,25,NaN,NaN
1,Petri Net,89,116611,25,70.0,103.0


Ringkasan model berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\06_process_discovery_model_summary.csv


## Cell 14 — Cek file gambar tersimpan

In [15]:
for file in [
    PROCESS_TREE_PATH,
    PETRI_NET_PATH
]:
    print(file.name, "ada:", file.exists())

06_process_tree_inductive_miner.png ada: True
06_petri_net_inductive_miner.png ada: True


## Cell 15 — Buat main learning log untuk process discovery utama

In [16]:
main_learning_activities = [
    "Course Viewed",
    "Material Viewed",
    "Quiz Module Viewed",
    "Quiz Access Prevented",
    "Quiz Started",
    "Quiz Viewed",
    "Quiz Updated",
    "Quiz Auto-saved",
    "Quiz Summary Viewed",
    "Quiz Submitted"
]

main_learning_df = pm_df[
    pm_df["concept:name"].isin(main_learning_activities)
].copy()

main_learning_df = main_learning_df.sort_values(
    by=["case:concept:name", "time:timestamp"]
).reset_index(drop=True)

print("Ukuran pm_df lengkap:", pm_df.shape)
print("Ukuran main_learning_df:", main_learning_df.shape)

print("\nJumlah case main learning:", main_learning_df["case:concept:name"].nunique())
print("Jumlah activity main learning:", main_learning_df["concept:name"].nunique())

display(main_learning_df["concept:name"].value_counts())

Ukuran pm_df lengkap: (116611, 5)
Ukuran main_learning_df: (113803, 5)

Jumlah case main learning: 89
Jumlah activity main learning: 10


concept:name
Quiz Viewed              42249
Quiz Updated             41209
Quiz Module Viewed        7348
Course Viewed             6384
Quiz Auto-saved           4594
Material Viewed           3405
Quiz Summary Viewed       2362
Quiz Started              2259
Quiz Submitted            2210
Quiz Access Prevented     1783
Name: count, dtype: Int64

## Cell 16 — Convert main learning dataframe ke event log PM4Py

In [17]:
main_learning_event_log = pm4py.convert_to_event_log(main_learning_df)

print("Main learning event log berhasil dibuat")
print("Jumlah trace/case:", len(main_learning_event_log))

Main learning event log berhasil dibuat
Jumlah trace/case: 89


## Cell 17 — Discovery process tree main learning

In [18]:
main_process_tree = pm4py.discover_process_tree_inductive(main_learning_event_log)

print("Main learning process tree berhasil ditemukan.")
print(main_process_tree)

Main learning process tree berhasil ditemukan.
+( *( 'Quiz Submitted', tau ), X( tau, *( 'Quiz Auto-saved', tau ) ), *( 'Quiz Summary Viewed', tau ), *( 'Quiz Viewed', tau ), *( 'Quiz Updated', tau ), *( 'Course Viewed', tau ), *( 'Quiz Started', tau ), *( 'Quiz Module Viewed', tau ), X( tau, *( 'Quiz Access Prevented', tau ) ), X( tau, *( 'Material Viewed', tau ) ) )


## Cell 18 — Simpan process tree main learning

In [19]:
MAIN_PROCESS_TREE_PATH = OUTPUT_FIGURES_DIR / "06_process_tree_main_learning_inductive_miner.png"

pm4py.save_vis_process_tree(
    main_process_tree,
    MAIN_PROCESS_TREE_PATH
)

print("Gambar main learning process tree berhasil disimpan ke:")
print(MAIN_PROCESS_TREE_PATH)

Gambar main learning process tree berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\figures\06_process_tree_main_learning_inductive_miner.png


## Cell 19 — Discovery Petri net main learning

In [20]:
main_net, main_initial_marking, main_final_marking = pm4py.discover_petri_net_inductive(
    main_learning_event_log
)

print("Main learning Petri net berhasil ditemukan.")
print("Jumlah places:", len(main_net.places))
print("Jumlah transitions:", len(main_net.transitions))
print("Initial marking:", main_initial_marking)
print("Final marking:", main_final_marking)

Main learning Petri net berhasil ditemukan.
Jumlah places: 28
Jumlah transitions: 31
Initial marking: ['source:1']
Final marking: ['sink:1']


## Cell 20 — Simpan Petri net main learning

In [21]:
MAIN_PETRI_NET_PATH = OUTPUT_FIGURES_DIR / "06_petri_net_main_learning_inductive_miner.png"

pm4py.save_vis_petri_net(
    main_net,
    main_initial_marking,
    main_final_marking,
    MAIN_PETRI_NET_PATH
)

print("Gambar main learning Petri net berhasil disimpan ke:")
print(MAIN_PETRI_NET_PATH)

Gambar main learning Petri net berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\figures\06_petri_net_main_learning_inductive_miner.png


## Cell 21 — Bandingkan model lengkap dan main learning

In [23]:
model_comparison = pd.DataFrame({
    "model": [
        "Full Compact Log - Petri Net",
        "Main Learning Log - Petri Net"
    ],
    "jumlah_case": [
        pm_df["case:concept:name"].nunique(),
        main_learning_df["case:concept:name"].nunique()
    ],
    "jumlah_event": [
        len(pm_df),
        len(main_learning_df)
    ],
    "jumlah_activity": [
        pm_df["concept:name"].nunique(),
        main_learning_df["concept:name"].nunique()
    ],
    "jumlah_places": [
        len(net.places),
        len(main_net.places)
    ],
    "jumlah_transitions": [
        len(net.transitions),
        len(main_net.transitions)
    ]
})

display(model_comparison)

MODEL_COMPARISON_PATH = OUTPUT_TABLES_DIR / "06_model_comparison_full_vs_main_learning.csv"
model_comparison.to_csv(MODEL_COMPARISON_PATH, index=False)

print("Perbandingan model berhasil disimpan ke:")
print(MODEL_COMPARISON_PATH)

,model,jumlah_case,jumlah_event,jumlah_activity,jumlah_places,jumlah_transitions
0,Full Compact Log - Petri Net,89,116611,25,70,103
1,Main Learning Log - Petri Net,89,113803,10,28,31


Perbandingan model berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\06_model_comparison_full_vs_main_learning.csv


## Kesimpulan Process Discovery dengan Inductive Miner

Pada tahap ini, event log hasil compacting digunakan untuk melakukan process discovery menggunakan algoritma Inductive Miner pada PM4Py. Event log yang digunakan telah berada dalam format standar process mining, yaitu `case:concept:name` sebagai case ID mahasiswa, `concept:name` sebagai aktivitas, dan `time:timestamp` sebagai waktu terjadinya aktivitas.

Hasil konversi menunjukkan bahwa event log berhasil dibaca oleh PM4Py sebagai 89 trace, yang merepresentasikan 89 mahasiswa. Process discovery pertama dilakukan pada compact log lengkap yang terdiri dari 116.611 event dan 25 aktivitas unik. Dari log tersebut, Inductive Miner berhasil menghasilkan process tree dan Petri net. Model Petri net lengkap memiliki 70 places dan 103 transitions, yang menunjukkan bahwa proses belajar mahasiswa pada LMS memiliki variasi aktivitas dan urutan yang cukup kompleks.

Karena model lengkap masih memiliki kompleksitas visual yang tinggi, process discovery juga dilakukan pada main learning log. Main learning log difokuskan pada aktivitas utama yang berkaitan dengan akses course, material, dan pengerjaan kuis, yaitu Course Viewed, Material Viewed, Quiz Module Viewed, Quiz Access Prevented, Quiz Started, Quiz Viewed, Quiz Updated, Quiz Auto-saved, Quiz Summary Viewed, dan Quiz Submitted. Subset ini tetap mencakup 89 mahasiswa dengan 113.803 event dan 10 aktivitas unik.

Hasil discovery pada main learning log menghasilkan Petri net yang lebih sederhana, yaitu 28 places dan 31 transitions. Penurunan kompleksitas ini menunjukkan bahwa pemilihan aktivitas utama membantu menghasilkan model proses yang lebih mudah dibaca dan diinterpretasikan tanpa menghilangkan pola dominan dari event log. Model ini memperlihatkan bahwa proses belajar mahasiswa pada LMS didominasi oleh aktivitas kuis, terutama siklus antara Quiz Viewed dan Quiz Updated, serta aktivitas pendukung seperti akses course, akses material, dan akses modul kuis.

Dengan demikian, hasil process discovery menunjukkan bahwa pola belajar mahasiswa tidak berbentuk alur linear sederhana, melainkan memiliki banyak variasi dan pengulangan aktivitas. DFG digunakan untuk membaca pola transisi utama secara lebih intuitif, sedangkan process tree dan Petri net digunakan sebagai model proses formal hasil algoritma Inductive Miner.
